# Simulate service operations

Simulate passenger requests and vehicle assignments on the packaged road network. The example compares two service areas and fleets of two to five vehicles, with four seats per vehicle. Review capacity, waiting-time, ride-time and depot assumptions before applying the model to another study. Results describe event-driven scenario operations, rather than a replay of recorded 30-second trajectories.

Select a Python kernel with the required packages; use the dependency profile in requirements.txt for the saved demand model. The reference installation uses the **ArcPy** kernel (Anaconda base). Run the cells in order. Shared settings are in `desktop_settings.json`; the default result folder is `../outputs/desktop_run_en`. Run the first setup cell after reopening an updated notebook so the helper code is refreshed. The scripts check dependencies without installing packages. Starting validation or a complete run preserves incompatible previous results in their original folder and selects a fresh run ID when required.

In [ ]:
from pathlib import Path
import sys
# Open notebooks within the toolkit; editor scripts can also locate their own folder.
_entry_dir = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
ROOT = next((p for p in [_entry_dir, *_entry_dir.parents, Path.cwd(), *Path.cwd().parents]
    if (p/'desktop_scripts/desktop_helpers.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Open the notebook from inside the toolkit folder, then rerun this cell.')
sys.path.insert(0, str(ROOT/'desktop_scripts'))
import importlib
import desktop_helpers
importlib.reload(desktop_helpers)
from desktop_helpers import check_environment, run, run_r, get_output_dir, get_study_config
import pandas as pd
versions = check_environment()
OUT = get_output_dir()

Inspect the eight scenarios. Complete notebook 04 first.

In [ ]:
import json
cfg = get_study_config()
print(pd.DataFrame(cfg['simulation']['scenarios']).to_string(index=False))
print('Replications per scenario:',cfg['simulation']['replications'])

Simulate all scenarios using the road network and forecast demand. Source: drt_toolkit/simulation.py.

In [ ]:
OUT = run('simulate')
summary = pd.read_csv(OUT/'simulation_summary.csv')
print(summary.to_string(index=False))

Review passenger conservation and replications.

In [ ]:
daily = pd.read_csv(OUT/'simulation_daily.csv')
assert (daily.requests == daily.served + daily.rejected).all()
print('Daily scenario-replication rows:',len(daily))
print(daily.groupby('ServiceID')[['requests','served','rejected','vkt_km']].mean().to_string())